# Synthetic Data Generator Validation

**Purpose:** Before scaling up generation, test each generator prompt on a small corpus sample and compare the output against real benchmark examples.

For each generator, this notebook shows:
1. **Benchmark reference** — What a real eval example looks like (RACE, BoolQ, GSM8K, etc.)
2. **Our generator prompt** — The prompt template from `03_SYNTHETIC_DATA_GENERATORS.md`
3. **API output** — What GPT-4o-mini actually produces from a corpus sample
4. **Comparison** — Does our output match the target format and quality?

Covers generators: A (Factual), C (Reading Comprehension), D (Temporal), E (Quantitative), H (Anti-Hallucination)

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import display, Markdown, HTML

# Project paths
PROJECT_ROOT = Path.cwd().parent.parent  # hist_LLM/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from openai import OpenAI

# Load API key
api_key = (PROJECT_ROOT / "key.txt").read_text().strip()
client = OpenAI(api_key=api_key)

MODEL = "gpt-4o-mini"
PERIOD_END_YEAR = 1999  # Using 1950-1999 as test period

def call_api(prompt, temperature=0.7, max_tokens=4096):
    """Call OpenAI API and return parsed JSON."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return json.loads(response.choices[0].message.content)

def show_json(data, title="Output"):
    """Pretty-print JSON output."""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])

print(f"Model: {MODEL}")
print(f"Test period end year: {PERIOD_END_YEAR}")
print("Ready.")

## Load Corpus Sample

Load a small sample of documents from the corpus. We use 3 diverse documents to test generators.

In [ ]:
import pandas as pd

DATA_ROOT = Path("D:/hist_LLM")

# Try loading from actual corpus; fall back to hardcoded samples
sample_texts = {}

try:
    # Load a few documents from different sources
    synth_input = DATA_ROOT / "periods" / "1950_1999" / "posttraining_data" / "synthetic" / "input"
    
    for source in ["economist", "nyt_filtered", "ft"]:
        pq = synth_input / f"{source}.parquet"
        if pq.exists():
            df = pd.read_parquet(pq)
            # Pick a doc with decent length
            good = df[df["text"].str.len() > 2000].head(3)
            if len(good) > 0:
                row = good.iloc[0]
                sample_texts[source] = row["text"][:6000]
                print(f"Loaded {source}: {len(row['text']):,} chars (truncated to 6000)")
    
    if not sample_texts:
        raise FileNotFoundError("No parquet files found")
        
except Exception as e:
    print(f"Could not load from D: drive ({e}). Using hardcoded samples.")
    # Fallback: paste a sample document here for testing
    sample_texts["economist"] = """PASTE A SAMPLE ECONOMIST ARTICLE HERE FOR TESTING.
    
Replace this placeholder with an actual document from the corpus.
Ideal: 2000-6000 chars, containing dates, numbers, and historical events."""

print(f"\nLoaded {len(sample_texts)} sample document(s): {list(sample_texts.keys())}")

---
## Generator A: Factual QA

### Benchmark Reference: MMLU / ARC-Challenge

**MMLU** (nanochat format via `render_mc()`):
```
Multiple Choice question: What is the longest river in Africa?
- Nile=A
- Congo=B
- Niger=C
- Zambezi=D

Respond only with the letter of the correct answer.
```
Model outputs: `A`

**ARC-Challenge** (same MC format):
```
Multiple Choice question: Which property of a mineral can be determined just by looking at it?
- luster=A
- mass=B
- weight=C
- hardness=D

Respond only with the letter of the correct answer.
```

**Our Generator A** currently produces open-ended QA. Below we test both the existing open-ended format AND a new MC variant.

In [ ]:
# --- Generator A: Existing open-ended format ---
text = list(sample_texts.values())[0]

QA_PROMPT = """Create 3 question-answer pairs from this text for LLM training.

Rules:
1. Questions must require analytical thinking, not just fact lookup
2. Answers must be directly supported by the text
3. Vary question types: cause-effect, comparison, analysis, inference, summary
4. Return a JSON object with key "qa_pairs" containing an array:

{{"qa_pairs": [{{"question": "Question 1?", "answer": "Answer 1."}}, {{"question": "Question 2?", "answer": "Answer 2."}}]}}

Text:
{text}"""

result_a = call_api(QA_PROMPT.format(text=text))
show_json(result_a, "Generator A: Factual QA (open-ended)")

In [ ]:
# --- Generator A: MC variant (to match MMLU/ARC format) ---

QA_MC_PROMPT = """Create 3 multiple-choice questions from this text for LLM evaluation.

Rules:
1. Each question must have exactly 4 options (A, B, C, D)
2. Exactly one option must be correct
3. Wrong options should be plausible but clearly incorrect
4. Questions should require analytical thinking, not just fact lookup

Return a JSON object:
{{"mc_questions": [
  {{"question": "...", "options": ["option A", "option B", "option C", "option D"], "correct": 0, "explanation": "..."}}
]}}

Text:
{text}"""

result_a_mc = call_api(QA_MC_PROMPT.format(text=text))
show_json(result_a_mc, "Generator A: Factual QA (MC variant)")

# Show how it would render in nanochat format
print("\n--- nanochat rendering ---")
for q in result_a_mc.get("mc_questions", [])[:1]:
    letters = ["A", "B", "C", "D"]
    rendered = f"Multiple Choice question: {q['question']}\n"
    for letter, opt in zip(letters, q['options']):
        rendered += f"- {opt}={letter}\n"
    rendered += "\nRespond only with the letter of the correct answer."
    print(rendered)
    print(f"\nCorrect: {letters[q['correct']]}")

---
## Generator C: Reading Comprehension

### Benchmark Reference: RACE

**RACE** (nanochat format):
```
Multiple Choice question: Read the following passage and answer the question.

Passage: The oil embargo imposed by OAPEC in 1973 had far-reaching consequences
for Western economies. Crude oil prices quadrupled from $3 to $12 per barrel
within months, triggering stagflation — simultaneous high inflation and economic
stagnation — across Europe and North America...

What was the primary economic consequence of the 1973 oil embargo according to the passage?
- Stagflation across Western economies=A
- Collapse of the OPEC cartel=B
- Rapid industrialization of oil-producing nations=C
- Immediate shift to renewable energy sources=D

Respond only with the letter of the correct answer.
```

### Benchmark Reference: BoolQ

**BoolQ** (nanochat format):
```
Read the following passage and answer True or False.

Passage: The European Economic Community was established by the Treaty of Rome
in 1957. Initially comprising six member states — Belgium, France, Germany, Italy,
Luxembourg, and the Netherlands — it aimed to create a common market...

Question: Was the EEC established before 1960?
```
Model outputs: `True`

**Key difference from Generator A:** The passage is INCLUDED in the training example.

In [ ]:
# --- Generator C: Reading Comprehension ---

RC_PROMPT = """Read the following passage carefully and create 3 reading comprehension
questions with answers.

Requirements:
1. Questions should require understanding the passage, not just keyword matching
2. Include a mix of question types:
   - Extractive: "According to the passage, what/who/when..."
   - Numerical: "How many..." / "What percentage..."
   - Inferential: "What can be inferred about..."
   - Comparative: "How does X compare to Y in the passage?"
3. Answers must be directly supported by the passage text
4. For each question, also create 3 wrong options for a multiple-choice version

Return a JSON object:
{{"rc_pairs": [
  {{"question": "...",
    "answer": "...",
    "type": "extractive|numerical|inferential|comparative",
    "mc_options": ["correct answer", "wrong 1", "wrong 2", "wrong 3"]}}
]}}

Passage:
{text}"""

result_c = call_api(RC_PROMPT.format(text=text))
show_json(result_c, "Generator C: Reading Comprehension")

# Show RACE-style rendering
print("\n--- RACE-style nanochat rendering ---")
for q in result_c.get("rc_pairs", [])[:1]:
    passage_preview = text[:500] + "..."
    rendered = f"Multiple Choice question: Read the following passage and answer the question.\n\n"
    rendered += f"Passage: {passage_preview}\n\n{q['question']}\n"
    letters = ["A", "B", "C", "D"]
    for letter, opt in zip(letters, q.get('mc_options', [])):
        rendered += f"- {opt}={letter}\n"
    rendered += "\nRespond only with the letter of the correct answer."
    print(rendered[:1000])

---
## Generator D: Temporal Reasoning

### Benchmark Reference: TimE (arXiv:2505.12891)

**Level 1 — Date extraction:**
```
Q: When was the North Atlantic Treaty signed?
A: April 4, 1949
```

**Level 2 — Relative reasoning:**
```
Q: What major international organization was established in the decade
   before the Korean War began?
A: The United Nations (1945), approximately 5 years before the Korean War (1950)
```

**Level 3 — Timeline ordering:**
```
Q: Arrange in chronological order: Marshall Plan, NATO formation,
   Berlin Blockade, Korean War
A: Marshall Plan (1948) -> Berlin Blockade (1948-49) -> NATO (1949) -> Korean War (1950)
```

**No existing dataset covers all 3 levels from a historical corpus. This is unique to our project.**

In [ ]:
# --- Generator D Level 1: Basic Temporal (single document) ---

TEMPORAL_L1_PROMPT = """From the following historical document, create 3 questions
that test basic temporal understanding.

For each question, also specify the FORMAT from: mc, cot, true_false, ranking

Required question subtypes (include at least one of each):
- Date/time extraction: "When did [event] happen?" -> format: mc or true_false
- Duration computation: "How long did [period/event] last?" -> format: cot
- Simple ordering: "Did [A] happen before or after [B]?" -> format: true_false or mc

Each question MUST have a definitive answer derivable from the text.

Return JSON:
{{"temporal_qa": [
  {{"question": "...",
    "answer": "...",
    "format": "mc|cot|true_false",
    "level": 1,
    "subtype": "extraction|duration|ordering",
    "mc_options": ["A", "B", "C", "D"]
  }}
]}}

Document:
{text}"""

result_d1 = call_api(TEMPORAL_L1_PROMPT.format(text=text))
show_json(result_d1, "Generator D Level 1: Basic Temporal")

# Show different format renderings
print("\n--- Format rendering examples ---")
for q in result_d1.get("temporal_qa", []):
    fmt = q.get("format", "open_ended")
    print(f"\n[{fmt.upper()}] {q.get('subtype', 'unknown')}")
    if fmt == "mc" and q.get("mc_options"):
        print(f"Q: {q['question']}")
        for letter, opt in zip('ABCD', q['mc_options']):
            print(f"  {letter}) {opt}")
    elif fmt == "true_false":
        print(f"Q: {q['question']}")
        print(f"A: {q['answer']}")
    elif fmt == "cot":
        print(f"Q: {q['question']}")
        print(f"A: {q['answer']}")

In [ ]:
# --- Generator D Level 2: Cross-Document (uses 2 source documents) ---
# This requires 2 documents. If we have multiple sources, use them.

sources = list(sample_texts.keys())
if len(sources) >= 2:
    text1 = sample_texts[sources[0]]
    text2 = sample_texts[sources[1]]
    source1, source2 = sources[0], sources[1]
else:
    # Split single document in half as fallback
    full = list(sample_texts.values())[0]
    mid = len(full) // 2
    text1, text2 = full[:mid], full[mid:]
    source1, source2 = "document_part1", "document_part2"

TEMPORAL_L2_PROMPT = """Given these two passages from the same historical period,
create 2 questions that require connecting information across them.

For each question, specify FORMAT from: cot, ranking

Required question subtypes:
- Contemporaneous: "What was happening in [domain B] while [event A] occurred?"
- Causal: "How might [earlier event in Passage 1] have influenced [later event in Passage 2]?"
- Period characterization: "Based on both passages, what defined the [decade]?"

Passage 1 ({source1}):
{text1}

Passage 2 ({source2}):
{text2}

Return JSON:
{{"temporal_qa": [
  {{"question": "...",
    "answer": "...",
    "format": "cot|ranking",
    "level": 2,
    "subtype": "contemporaneous|causal|characterization",
    "reasoning": "Step-by-step reasoning connecting the two passages..."
  }}
]}}"""

result_d2 = call_api(TEMPORAL_L2_PROMPT.format(
    source1=source1, text1=text1[:3000],
    source2=source2, text2=text2[:3000]
))
show_json(result_d2, "Generator D Level 2: Cross-Document Temporal")

---
## Generator E: Corpus-Grounded Quantitative Reasoning

### Benchmark Reference: GSM8K

**GSM8K format:**
```
Q: Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning
   and bakes muffins for her friends every day with four. She sells every remaining
   egg at the farmers' market daily for $2 per egg. How much in dollars does she
   make every day at the farmers' market?

A: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
   She makes 9 * 2 = <<9*2=18>>$18 every day.
   #### 18
```

**Key difference:** GSM8K uses invented numbers. Our Generator E uses REAL numbers from historical documents (trade figures, economic data, demographics).

**Pre-filter:** Only send document chunks that contain sufficient numerical content.

In [ ]:
import re

def has_sufficient_numbers(text, min_numbers=3):
    """Check if text has enough numerical content for math problems."""
    patterns = [
        r'\$[\d,.]+\s*(million|billion|thousand)?',   # currency
        r'[\d,.]+\s*%',                                # percentages
        r'[\d,.]+\s*(million|billion|thousand|tons|bushels|barrels)',  # quantities
        r'\b\d{1,3}(,\d{3})+\b',                     # large numbers
    ]
    matches = sum(len(re.findall(p, text, re.IGNORECASE)) for p in patterns)
    return matches, matches >= min_numbers

# Check which samples have enough numbers
print("Numeric content check:")
best_text = None
for source, txt in sample_texts.items():
    count, sufficient = has_sufficient_numbers(txt)
    status = "OK" if sufficient else "LOW"
    print(f"  {source}: {count} numeric patterns [{status}]")
    if sufficient and best_text is None:
        best_text = txt

if best_text is None:
    best_text = list(sample_texts.values())[0]
    print("\n  Warning: No sample has 3+ numeric patterns. Using first sample anyway.")

In [ ]:
# --- Generator E: Quantitative Reasoning ---

QUANTITATIVE_PROMPT = """From the following historical document containing numerical
data, create 3 math word problems that require step-by-step calculation.

Requirements:
1. Use ONLY numbers that actually appear in the text (prices, percentages,
   populations, trade volumes, etc.)
2. Each problem should require 2-4 calculation steps
3. Show complete step-by-step solutions
4. For each problem, specify FORMAT from: open_ended, cot, fill_blank

Problem types to vary across:
- Percentage change: "By what percentage did X change from Y to Z?"
- Compound growth: "If X grew at Y% annually, what was the value after N years?"
- Ratio comparison: "What was the ratio of X to Y?"
- Difference: "How much more/less was X compared to Y?"

Return JSON:
{{"math_problems": [
  {{"problem": "...",
    "solution": "Step 1: ...\\nStep 2: ...\\nStep 3: ...",
    "answer": "...",
    "format": "open_ended|cot|fill_blank",
    "source_numbers": ["$240 million", "8%"]
  }}
]}}

Historical document:
{text}"""

result_e = call_api(QUANTITATIVE_PROMPT.format(text=best_text))
show_json(result_e, "Generator E: Quantitative Reasoning")

# Show different format renderings
print("\n--- Format comparisons ---")
for p in result_e.get("math_problems", []):
    fmt = p.get("format", "open_ended")
    print(f"\n[{fmt.upper()}]")
    if fmt == "cot":
        print(f"Q: {p['problem']}")
        print(f"A: <think>\n{p['solution']}\n</think>\n{p['answer']}")
    elif fmt == "fill_blank":
        print(f"Q: {p['problem']}")
        print(f"A: {p['answer']}")
    else:
        print(f"Q: {p['problem']}")
        print(f"A: {p['solution']}\nFinal: {p['answer']}")
    print(f"Source numbers: {p.get('source_numbers', [])}")

---
## Generator H: Anti-Hallucination

### Why This Generator Exists

**Problem:** LLMs hallucinate by default. If asked about something outside their training period, they fabricate answers rather than refusing.

**Academic backing:**
- **R-Tuning** (Zhang et al., NeurIPS 2023): Training with refusal examples significantly reduces hallucination
- **Gekhman et al.** (EMNLP 2024): SFT on facts not seen in pretraining *increases* hallucination — so fact memorization via SFT backfires

**Generator H teaches the model WHERE ITS KNOWLEDGE STOPS** — not what it knows (that's Generator A + pretraining).

This generator does NOT use corpus text as input. It generates questions about post-period events and trains the model to refuse appropriately.

In [ ]:
# --- Generator H Subtype 1: Direct Post-Period Questions ---

ANTI_HALLUC_DIRECT_PROMPT = """Generate 5 question-answer pairs where the question
asks about events, technologies, or developments that occurred AFTER the year {end_year}.

For each pair, specify FORMAT from: mc, open_ended, true_false

The correct response must ALWAYS:
1. Acknowledge the question
2. State clearly that the information is beyond the model's knowledge period
3. Offer what WAS known as of {end_year} (if relevant context exists)
4. VARY the refusal phrasing across pairs

Topics to cover (one per pair):
- Technology invented after {end_year}
- Political event after {end_year}
- Cultural phenomenon after {end_year}
- Scientific discovery after {end_year}
- Company or organization founded after {end_year}

For MC format: one option should be "This occurred after my knowledge period."
For T/F: make a statement about a post-period fact; answer should explain inability to verify.

Return JSON:
{{"anti_halluc_pairs": [
  {{"question": "...",
    "answer": "...",
    "format": "mc|open_ended|true_false",
    "event_year": 2005,
    "domain": "technology|politics|culture|science|economics",
    "mc_options": ["opt A", "opt B", "opt C", "opt D"]
  }}
]}}"""

result_h1 = call_api(ANTI_HALLUC_DIRECT_PROMPT.format(end_year=PERIOD_END_YEAR))
show_json(result_h1, f"Generator H: Anti-Hallucination (post-{PERIOD_END_YEAR})")

# Show format examples
print("\n--- Training example renderings ---")
for pair in result_h1.get("anti_halluc_pairs", [])[:3]:
    fmt = pair.get("format", "open_ended")
    print(f"\n[{fmt.upper()}] Domain: {pair.get('domain', '?')} | Event year: {pair.get('event_year', '?')}")
    print(f"User: {pair['question'][:200]}")
    print(f"Asst: {pair['answer'][:300]}")

In [ ]:
# --- Generator H Subtype 2: Boundary Probing ---

BOUNDARY_PROBE_PROMPT = """Generate 3 questions that probe the boundary of knowledge
at year {end_year}.

These should be questions where:
- A partial answer exists within the period (ongoing trend, unresolved issue)
- A full/updated answer requires post-{end_year} knowledge
- The model should provide the partial answer AND explicitly note the limitation

The answer should follow this pattern:
"As of {end_year}, [what was known/happening]. [The situation was still developing /
The outcome had not yet been determined / Further developments occurred after my
knowledge period.]"

Example for end_year=1999: Asking about the outcome of the Kosovo conflict.

Return JSON:
{{"boundary_pairs": [
  {{"question": "...",
    "answer": "...",
    "format": "open_ended",
    "context_note": "What actually happened after {end_year} (not shown to model)"
  }}
]}}"""

result_h2 = call_api(BOUNDARY_PROBE_PROMPT.format(end_year=PERIOD_END_YEAR))
show_json(result_h2, f"Generator H: Boundary Probing (at {PERIOD_END_YEAR})")

# Show examples with the hidden context
print("\n--- Boundary probe examples ---")
for pair in result_h2.get("boundary_pairs", []):
    print(f"\nQ: {pair['question'][:200]}")
    print(f"A: {pair['answer'][:300]}")
    print(f"[Hidden context: {pair.get('context_note', 'N/A')[:200]}]")

---
## Summary: Generator Output vs. Benchmark Format

After running the cells above, compare:

| Generator | Target Benchmark | Key Check |
|-----------|-----------------|----------|
| A (open-ended) | General SFT | Are questions analytical? Do answers cite the text? |
| A (MC variant) | MMLU / ARC | Do options render correctly in nanochat `render_mc()` format? |
| C | RACE / BoolQ | Is the passage included? Are question types varied? |
| D Level 1 | TimE Level 1 | Are dates/durations/orderings extractable from source? |
| D Level 2 | TimE Level 2-3 | Do questions genuinely require BOTH passages? |
| E | GSM8K | Are source numbers real? Are solutions arithmetically correct? |
| H Direct | R-Tuning | Are refusals varied? Do they offer pre-period context? |
| H Boundary | R-Tuning | Does the model give partial answers + flag limitations? |

### Quality Checklist
- [ ] JSON parses correctly (no format errors)
- [ ] Questions are non-trivial (not just keyword lookup)
- [ ] MC options are plausible (not obviously wrong)
- [ ] Temporal questions actually require temporal reasoning
- [ ] Math problems use REAL numbers from the source text
- [ ] Refusal phrasings are diverse (not repetitive)
- [ ] Boundary probes give partial answers (not just refusals)

In [ ]:
# --- Quick stats ---
results = {
    "A (open-ended)": result_a.get("qa_pairs", []),
    "A (MC)": result_a_mc.get("mc_questions", []),
    "C (RC)": result_c.get("rc_pairs", []),
    "D L1 (temporal)": result_d1.get("temporal_qa", []),
    "D L2 (cross-doc)": result_d2.get("temporal_qa", []),
    "E (quantitative)": result_e.get("math_problems", []),
    "H (direct refusal)": result_h1.get("anti_halluc_pairs", []),
    "H (boundary)": result_h2.get("boundary_pairs", []),
}

print("Generation Summary")
print("=" * 50)
total = 0
for name, items in results.items():
    n = len(items)
    total += n
    print(f"  {name}: {n} examples")
print(f"\n  Total: {total} examples generated")
print(f"\nReview the outputs above and check the quality checklist.")
print(f"If prompts need tuning, edit them in the cells above and re-run.")